# 3901161_1386_EasyTSLite.csv

This CSV is an Argo profile file with a metadata comment block at the top and tabular measurements below.

## File Structure

The first lines begin with `#`. These are comments used as metadata, not data rows. They describe the profile and the source of the file.

In this file, the metadata includes:

- format: `EasyOneArgoTSLite`
- creation date: `2026-03-16T04:30:37Z`
- creation centre: `Ifremer`
- data source DOI: `https://doi.org/10.17882/42182#126945`
- platform number: `3901161`
- cycle number: `1386`
- profile date: `2017-02-09T23:29:42Z`
- profile latitude: `6.144`
- profile longitude: `-119.519`

The comment lines also define the variables used in the profile:

- pressure = sea water pressure, measured in decibar
- temperature = sea temperature in-situ on the ITS-90 scale
- salinity = practical salinity

After the comments, the file switches to a comma-separated header row:

- pressure (decibar)
- temperature (degree_celsius)
- salinity (dimensionless)
- pressure_error (decibar)
- temperature_error (degree_celsius)
- salinity_error (dimensionless)

Each following row is one measurement at a given pressure level.

## How To Read It In Pandas

Because the metadata lines start with `#`, pandas should skip them when reading the file:

```python
single_argo = pd.read_csv('3901161_1386_EasyTSLite.csv', comment='#')
```

That keeps the header row and measurement rows while ignoring the comment block.

## Notes

- The parser error happens if pandas tries to interpret the comment block as tabular data.
- The file is comma-separated, so the default delimiter is correct once the comment lines are skipped.

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re


csv_path = Path('3901161_1386_EasyTSLite.csv')
argo_metadata = {}
with csv_path.open('r', encoding='utf-8', errors='replace') as csv_file:
    for line in csv_file:
        if not line.startswith('#'):
            break
        comment = line[1:].strip()
        if '=' in comment:
            key, value = comment.split('=', 1)
            argo_metadata[key.strip()] = value.strip()
        else:
            parts = comment.split(None, 1)
            if len(parts) == 2:
                argo_metadata[parts[0].strip()] = parts[1].strip()
            else:
                argo_metadata[comment] = None

single_argo = pd.read_csv(csv_path, comment='#')

# Add one constant metadata column per key for every row in the profile table.
for key, value in argo_metadata.items():
    safe_key = re.sub(r'[^0-9a-zA-Z_]+', '_', key.strip().lower()).strip('_')
    meta_col = f"meta_{safe_key}" if safe_key else "meta_unknown"
    single_argo[meta_col] = value

single_argo.attrs['metadata'] = argo_metadata
single_argo.head()

,pressure (decibar),temperature (degree_celsius),salinity (dimensionless),pressure_error (decibar),temperature_error (degree_celsius),salinity_error (dimensionless),meta_format,meta_creation_date,meta_creation_centre,meta_creation_centre_pid,...,meta_platform_number,meta_cycle_number,meta_direction_of_profile,meta_data_mode,meta_profile_date,meta_profile_latitude,meta_profile_longitude,meta_pressure,meta_temperature,meta_salinity
0,2.0,27.643,34.076,2.42,0.003,0.01,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,1386,A,D,2017-02-09T23:29:42Z,6.144,-119.519,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
1,5.0,27.636,34.077,2.42,0.003,0.01,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,1386,A,D,2017-02-09T23:29:42Z,6.144,-119.519,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
2,10.0,27.528,34.076,2.42,0.003,0.01,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,1386,A,D,2017-02-09T23:29:42Z,6.144,-119.519,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
3,15.0,27.435,34.071,2.42,0.003,0.01,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,1386,A,D,2017-02-09T23:29:42Z,6.144,-119.519,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
4,20.0,27.424,34.071,2.42,0.003,0.01,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,1386,A,D,2017-02-09T23:29:42Z,6.144,-119.519,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity


In [18]:
from pathlib import Path
import pandas as pd
import re


def _parse_argo_metadata(csv_file):
    metadata = {}
    with Path(csv_file).open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if not line.startswith("#"):
                break
            comment = line[1:].strip()
            if not comment:
                continue
            if "=" in comment:
                key, value = comment.split("=", 1)
                metadata[key.strip()] = value.strip()
            else:
                parts = comment.split(None, 1)
                if len(parts) == 2:
                    metadata[parts[0].strip()] = parts[1].strip()
                else:
                    metadata[comment] = None
    return metadata


def combine_csvs_from_subfolders(
    root_folder,
    pattern="*.csv",
    recursive=True,
    read_kwargs=None,
    max_files=None,
    include_metadata=True,
):
    """
    Read CSV files from a folder (optionally including subfolders)
    and combine them into a single pandas DataFrame.

    Parameters
    ----------
    root_folder : str | Path
        Folder that contains CSV files directly or inside subfolders.
    pattern : str, default "*.csv"
        File name pattern to match.
    recursive : bool, default True
        If True, search all subfolders with rglob; otherwise only top-level files.
    read_kwargs : dict | None
        Optional keyword args passed to pd.read_csv (e.g., {"comment": "#"}).
    max_files : int | None
        Optional limit on number of files to read (useful for large folders).
    include_metadata : bool, default True
        If True, parse comment-block metadata and append as columns.

    Returns
    -------
    pd.DataFrame
        Concatenated DataFrame containing all matched CSV files.
    """
    root = Path(root_folder)
    if not root.exists() or not root.is_dir():
        raise ValueError(f"Invalid folder path: {root}")

    read_kwargs = read_kwargs or {}
    csv_files = sorted(root.rglob(pattern) if recursive else root.glob(pattern))

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found under: {root}")

    if max_files is not None:
        if max_files <= 0:
            raise ValueError("max_files must be a positive integer")
        csv_files = csv_files[:max_files]

    frames = []
    for csv_file in csv_files:
        df = pd.read_csv(csv_file, **read_kwargs)

        if include_metadata:
            metadata = _parse_argo_metadata(csv_file)
            for key, value in metadata.items():
                safe_key = re.sub(r"[^0-9a-zA-Z_]+", "_", key.strip().lower()).strip("_")
                meta_col = f"meta_{safe_key}" if safe_key else "meta_unknown"
                df[meta_col] = value

        frames.append(df)

    return pd.concat(frames, ignore_index=True)



In [ ]:
# Load the local 3901161 folder at project root.
single_agro = combine_csvs_from_subfolders(
    "3901161",
    read_kwargs={"comment": "#"},
    include_metadata=True,
)

single_argo.shape

meta_cols = [c for c in single_argo.columns if c.startswith("meta_")]
print("Rows:", len(single_argo))
print("Metadata columns:", len(meta_cols))
print("First metadata columns:", meta_cols[:8])
single_argo.head()

Rows: 148159
Metadata columns: 16
First metadata columns: ['meta_format', 'meta_creation_date', 'meta_creation_centre', 'meta_creation_centre_pid', 'meta_data_source_doi', 'meta_data_centre', 'meta_platform_number', 'meta_cycle_number']


,pressure (decibar),temperature (degree_celsius),salinity (dimensionless),pressure_error (decibar),temperature_error (degree_celsius),salinity_error (dimensionless),meta_format,meta_creation_date,meta_creation_centre,meta_creation_centre_pid,...,meta_platform_number,meta_cycle_number,meta_direction_of_profile,meta_data_mode,meta_profile_date,meta_profile_latitude,meta_profile_longitude,meta_pressure,meta_temperature,meta_salinity
0,2.0,23.643,34.387,2.42,0.004,0.010,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,0,A,D,2014-01-28T02:06:30Z,0.004,-112.001,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
1,5.0,23.583,34.408,2.42,0.004,0.016,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,0,A,D,2014-01-28T02:06:30Z,0.004,-112.001,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
2,10.0,22.863,34.474,2.42,0.004,0.010,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,0,A,D,2014-01-28T02:06:30Z,0.004,-112.001,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
3,15.0,22.484,34.520,2.42,0.004,0.018,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,0,A,D,2014-01-28T02:06:30Z,0.004,-112.001,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
4,20.0,21.707,34.628,2.42,0.004,0.013,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,0,A,D,2014-01-28T02:06:30Z,0.004,-112.001,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity


In [21]:
# make single agro into a csv file
single_argo.to_csv("single_argo.csv", index=False)
